# 01 — Data Profile & Quality
**Goal**: Understand raw data thoroughly before any analysis

**Derived Dataset**: Metadata tables (profile_report.json)

**HR Value**: Data trust and transparency

**Employee Value**: Confidence in data-driven decisions

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os
from pathlib import Path

# Determine project root
cwd = Path.cwd()
if (cwd / "data/raw/employee_data.csv").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "data/raw/employee_data.csv").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError("Cannot find project root containing data/raw/employee_data.csv")
os.chdir(PROJECT_ROOT)
PROJECT_ROOT = Path.cwd().resolve()  # now absolute

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.figsize"] = (10, 5)

RAW_PATH = PROJECT_ROOT / "data/raw/employee_data.csv"
ANALYSIS_DIR = PROJECT_ROOT / "data/analysis/01_data_profile"
FIGURES_DIR = PROJECT_ROOT / "reports/figures"
Path(FIGURES_DIR).mkdir(parents=True, exist_ok=True)

df = pd.read_csv(RAW_PATH)
print(f"Shape: {df.shape}")
print(f"Memory: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

## 1. Column Overview

In [ ]:
col_info = pd.DataFrame({
    "Column": df.columns,
    "Type": df.dtypes.values,
    "Non-Null": df.count().values,
    "Null": df.isnull().sum().values,
    "Null %": (df.isnull().mean() * 100).values.round(1),
    "Unique": [df[c].nunique() for c in df.columns],
})
col_info

In [ ]:
# Numeric summary
num_cols = df.select_dtypes(include=[np.number]).columns
df[num_cols].describe().T

In [ ]:
# Categorical summary
cat_cols = df.select_dtypes(include=["object"]).columns
for col in cat_cols:
    print(f"\n--- {col} ---")
    print(df[col].value_counts().to_string())

## 2. Distribution Plots

In [ ]:
# Numeric distribution plots
n_numeric = len(num_cols)
n_cols = min(n_numeric, 3)
n_rows = (n_numeric + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 4 * n_rows))
axes = axes.flatten() if n_numeric > 1 else [axes]
for i, col in enumerate(num_cols):
    if i < len(axes):
        sns.histplot(df[col], bins=30, ax=axes[i], kde=True)
        axes[i].set_title(f"{col} Distribution")
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/01_numeric_distributions.png", bbox_inches="tight")
plt.show()

In [ ]:
# Categorical distribution plots
key_cats = ["EmployeeStatus", "GenderCode", "DepartmentType", "RaceDesc",
            "Performance Score", "EmployeeType", "PayZone", "MaritalDesc"]
fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.flatten()
for i, col in enumerate(key_cats):
    if i < len(axes):
        df[col].value_counts().plot(kind="bar", ax=axes[i], title=col)
        axes[i].tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/01_categorical_distributions.png", bbox_inches="tight")
plt.show()

## 3. Missing Value Analysis

In [ ]:
# Missing value heatmap
fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(df.isnull(), cbar=False, cmap="viridis", ax=ax)
ax.set_title("Missing Value Heatmap")
plt.savefig(f"{FIGURES_DIR}/01_missing_heatmap.png", bbox_inches="tight")
plt.show()

In [ ]:
# Missing patterns by column
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(10, 5))
missing.plot(kind="bar", ax=ax, title="Missing Values by Column")
ax.set_ylabel("Count")
for i, v in enumerate(missing.values):
    ax.text(i, v + 5, f"{v/len(df)*100:.1f}%", ha="center", fontsize=9)
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/01_missing_by_column.png", bbox_inches="tight")
plt.show()

## 4. Date Parsing Analysis

In [ ]:
date_cols = ["StartDate", "ExitDate", "DOB"]
for col in date_cols:
    parsed = pd.to_datetime(df[col], format="mixed", errors="coerce")
    success = parsed.notna().sum()
    fail = parsed.isna().sum()
    print(f"{col}: {success}/{len(df)} parsed ({success/len(df)*100:.1f}%)")
    if col == "DOB" and fail > 0:
        print(f"  Sample failed DOB values: {df[col].iloc[parsed.isna().values].head(10).tolist()}")

## 5. Outlier Detection

In [ ]:
def detect_outliers_iqr(series):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return series[(series < lower) | (series > upper)]

for col in num_cols:
    outliers = detect_outliers_iqr(df[col])
    print(f"{col}: {len(outliers)} outliers ({len(outliers)/len(df)*100:.1f}%)")

## 6. Correlation Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
corr = df[num_cols].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=ax)
ax.set_title("Correlation Matrix")
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/01_correlation_matrix.png", bbox_inches="tight")
plt.show()

## 7. Key Insights & Data Quality Notes

In [ ]:
insights = {
    "total_records": len(df),
    "total_columns": len(df.columns),
    "missing_columns": list(df.columns[df.isnull().any()]),
    "highest_missing_col": df.columns[df.isnull().mean().argmax()],
    "highest_missing_pct": round(df.isnull().mean().max() * 100, 1),
    "n_unique_employees": df["EmpID"].nunique(),
    "date_columns_failing": [c for c in date_cols if pd.to_datetime(df[c], format="mixed", errors="coerce").isna().sum() > 0],
}
print("\n--- Key Data Quality Insights ---")
for k, v in insights.items():
    print(f"  {k}: {v}")

with open(f"{ANALYSIS_DIR}/insights_summary.json", "w") as f:
    json.dump(insights, f, indent=2, default=str)
print("\nSaved to analysis directory.")

In [ ]:
print("\nProfile complete. Key takeaways:")
print(f"- {len(df)} employees, {len(df.columns)} columns")
print(f"- {len(df.columns[df.isnull().any()])} columns have missing data")
print(f"- ExitDate and TerminationDescription: {df['ExitDate'].isnull().sum()} null (active employees)")
print(f"- DOB: {pd.to_datetime(df['DOB'], format='mixed', errors='coerce').isna().sum()} unparseable dates")